# MLLMs Know Where to Look in Google Colab

This notebook adapts the full [`saccharomycetes/mllms_know`](https://github.com/saccharomycetes/mllms_know) repo for a single-GPU Colab workflow.

It keeps the repo's original Python scripts, installs the custom `transformers` fork, prepares a supported dataset, runs inference, and computes evaluation scores.

The notebook is self-contained, so it does not rely on any extra helper files beyond the upstream repo itself.

Practical runtime guidance:
- `qwen2_5` is the safest Colab default.
- `llava` and `blip` are heavier and usually need a higher-memory GPU runtime.
- Some Hugging Face models may require a login token.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/saccharomycetes/mllms_know.git"
REPO_DIR = Path("/content/mllms_know")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/mllms_know
print("Working directory:", Path.cwd())

In [ ]:
MODEL = "qwen2_5"      # options: qwen2_5, llava, blip
METHOD = "rel_att"      # options include: rel_att, grad_att, pure_grad, *_high
TASK = "textvqa"        # easiest benchmark to wire up from the README
TOTAL_CHUNKS = 1         # increase if you want to split work sequentially
SAVE_PATH = "./data/results"
HF_TOKEN = ""           # set if the selected model requires Hugging Face auth

print({
    "MODEL": MODEL,
    "METHOD": METHOD,
    "TASK": TASK,
    "TOTAL_CHUNKS": TOTAL_CHUNKS,
    "SAVE_PATH": SAVE_PATH,
})

In [ ]:
import subprocess
import sys

base_packages = [
    "numpy<2",
    "scipy<1.13",
    "huggingface-hub<1.0",
    "accelerate>=0.26.0",
    "datasets>=3.0.0",
    "Pillow>=10.0.1",
    "scikit-image>=0.25.2",
    "matplotlib>=3.8.0",
    "pandas>=2.2.0",
    "sentencepiece",
    "tqdm>=4.27",
]

def run(cmd):
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "uninstall", "-y", "transformers", "tokenizers"])

if MODEL == "qwen2_5":
    run([
        sys.executable, "-m", "pip", "install", *base_packages,
        "transformers==4.50.0",
        "tokenizers>=0.21,<0.22",
        "qwen-vl-utils",
    ])
else:
    run([
        sys.executable, "-m", "pip", "install", *base_packages,
        "tokenizers>=0.20,<0.21",
        "qwen-vl-utils",
    ])
    run([sys.executable, "-m", "pip", "install", "-e", "./transformers", "--no-deps"])

print(f"Install complete for MODEL={MODEL}")
print("Restart the runtime after this cell finishes, then continue from runtime-check.")

In [ ]:
import os
import torch
import platform

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a GPU runtime in Colab: Runtime > Change runtime type > GPU.")

In [ ]:
if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("Hugging Face login complete.")
else:
    print("HF_TOKEN is empty. Continuing without login.")

In [ ]:
from pathlib import Path

run_py = Path("run.py")
text = run_py.read_text(encoding="utf-8")

if MODEL == "qwen2_5":
    old = '        max_pixels = 256 * 28 * 28\n        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(args.model_id, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).to(args.device)\n        processor = AutoProcessor.from_pretrained(args.model_id, max_pixels=max_pixels)\n        processor.image_processor.size["longest_edge"] = max_pixels # this is likely a bug in current transformers (4.50.0) library, passing in max_pixels to from_pretrained does not work\n'
    new = '        max_pixels = 128 * 28 * 28\n        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(args.model_id, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True).to(args.device)\n        processor = AutoProcessor.from_pretrained(args.model_id, max_pixels=max_pixels)\n        processor.image_processor.size["longest_edge"] = max_pixels # reduced for Colab GPU memory\n'
    if old in text:
        text = text.replace(old, new)
        run_py.write_text(text, encoding="utf-8")
        print("Patched run.py for lower Qwen2.5-VL max_pixels on Colab.")
    elif 'max_pixels = 128 * 28 * 28' in text:
        print("run.py is already patched for Colab.")
    else:
        print("Expected Qwen block not found in run.py; patch not applied.")
else:
    print("Skipping Qwen Colab patch because MODEL is not qwen2_5.")

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

def prepare_textvqa(base_dir: str = "data/textvqa") -> None:
    base = Path(base_dir)
    images_dir = base / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    annotation_path = base / "TextVQA_0.5.1_val.json"
    unified_path = base / "data.json"

    image_zip = images_dir / "train_val_images.zip"
    image_url = "https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip"
    ann_url = "https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json"

    if not image_zip.exists():
        print("Downloading TextVQA images...")
        urlretrieve(image_url, image_zip)

    marker = images_dir / "_extracted.ok"
    if not marker.exists():
        print("Extracting TextVQA images...")
        with zipfile.ZipFile(image_zip, "r") as zf:
            zf.extractall(images_dir)
        train_dir = images_dir / "train_images"
        if train_dir.exists():
            for item in train_dir.iterdir():
                target = images_dir / item.name
                if not target.exists():
                    shutil.move(str(item), str(target))
            shutil.rmtree(train_dir)
        marker.write_text("done", encoding="utf-8")

    if not annotation_path.exists():
        print("Downloading TextVQA annotations...")
        urlretrieve(ann_url, annotation_path)

    if not unified_path.exists():
        print("Creating repo-format TextVQA JSON...")
        with annotation_path.open("r", encoding="utf-8") as f:
            raw = json.load(f)

        unified = []
        for data_id, sample in enumerate(raw["data"]):
            unified.append({
                "id": str(data_id).zfill(10),
                "question": sample["question"],
                "labels": sample["answers"],
                "image_path": f"{sample['image_id']}.jpg",
            })

        with unified_path.open("w", encoding="utf-8") as f:
            json.dump(unified, f, indent=2)

    print("TextVQA is ready:", unified_path)


if TASK == "textvqa":
    prepare_textvqa()
else:
    print("This notebook auto-prepares TextVQA only. For other datasets, upload data into ./data/<task>/ and follow info.py conventions.")

In [ ]:
import subprocess
import sys

for chunk_id in range(TOTAL_CHUNKS):
    cmd = [
        sys.executable,
        "run.py",
        "--chunk_id", str(chunk_id),
        "--total_chunks", str(TOTAL_CHUNKS),
        "--model", MODEL,
        "--task", TASK,
        "--method", METHOD,
        "--save_path", SAVE_PATH,
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
!python get_score.py --data_dir {SAVE_PATH} --save_path ./

In [ ]:
from pathlib import Path
import json
import pandas as pd

report_json = Path("evaluation_report.json")
report_csv = Path("evaluation_report.csv")

if report_json.exists():
    with report_json.open("r", encoding="utf-8") as f:
        data = json.load(f)
    display(pd.DataFrame(data))
else:
    print("No evaluation report found yet.")

print("CSV path:", report_csv.resolve())